# Datamine COMBMOD Process Example

This notebook demonstrates how to configure and run the **`combmod`** process wrapper in `dmstudio`.

## Process Description

## COMBMOD

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Combine up to 20 block models that may have different parent cells sizes, origins and extents into a single block model.

**COMBMOD** is useful in situations where models that have been generated to model local regions need to be recombined with a larger resource model, or where multiple local models need to be combined to use as a single model for easier reporting or visualisation.

### The Prototype PROTO file

The input model prototype file is optional. If this file is specified then the output model will have the same origin, extents, parent cell size and rotation as this file. All records in this file are ignored, only the data definition is used. If you have a file that defines the required extents of the output model and also has records that need to be combined then you need to specify it as a prototype and as one of the input files.

### The first input model file IN1

If a prototype file is not specified then the first input file is used to determine the parent cell size of the output model. The origin and extents of the output model is determined from the minimum and maximum extents of all the input models. If a prototype file is not specified then origin of the output model will lie on a parent cell boundary of the first input model. If only two input models are specified and they both share the same model definition the process is equivalent to running the ADDMOD process.

### Model Superimposition Order

Models are superimposed on one another according to **ADDMOD** logic. The order in which the input models are specified is therefore important. Input model 2 is superimposed on input model 1. Input model 3 is then superimposed on the combination of 1 and 2. This process is continued for all input models.

Any input model containing zero records, unless it is used as the prototype or causes a rotation conflict, is ignored

### Rotated models

COMBMOD supports rotated models but will only combine models that all share the same model rotation parameters, i.e. models which have identical values for the implicit fields **ANGLE1** , **ANGLE2** , **ANGLE3** , **X0** , **Y0** , **Z0** , **ROTAXIS1** , **ROTAXIS2** and **ROTAXIS3**. If any of the input files do not share the same rotation parameters as either the prototype file, or the first input file if the prototype is not specified, then the process exits with an error.

If the prototype or first model is not rotated and any of the subsequent input files are rotated then the process exist with an error

### TOLERNCE Parameter

The @**TOLERNCE** parameter is used to define the smallest subcell that will be written to the output model file. It is defined as a factor of the parent cell size in the three dimensions. If a subcell has dimensions xs, ys, zs and the parent cell has dimensions xp, yp, zp, then if xs<@**TOLERNCE** *xp or ys<@**TOLERNCE** *yp or zs<@**TOLERNCE** *zp then the subcell is considered too small and will not be written to the output model. Care should be taken if the output model is a seam type model with just a single cell in one direction. In such a case it may be necessary to set a value which is smaller than the default of 0.001 to avoid subcells being eliminated unnecessarily.

### Input Files:

* **proto** (Model Prototype):
  Input block model prototype that defines the extents and parent cell size of the
  combined model. Records in this file are ignored. This file is optional. If it is not
  specified models will be combined to fill the volume covered by the range of all input
  files.
  Required=Yes

* **in1** (Model):
  First input model for combining (sorted on IJK). If no prototype is specified the
  output combined model prototype will have the same parent cell size specification as
  this file and the limits will be determined from the combined range of all input files.
  Required=Yes

* **in2** (Model):
  Second input model for combining (sorted on IJK)
  Required=Yes

* **in3** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in4** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in5** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in6** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in7** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in8** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in9** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in10** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in11** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in12** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in13** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in14** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in15** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in16** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in17** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in18** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in19** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

* **in20** (Model):
  Additional, optional input model files for combining (sorted on IJK)
  Required=No

### Output Files:

* **modelout** (Model):
  Output combined model
  Required=Yes

### Fields:

### Parameters:

* **tolernce**:
  Defines the smallest cell that will be included in OUT. Defined as a factor of XINC,
  YINC, ZINC. Default = (0.001).
  Range=Undefined
  Values=Undefined
  Default=0.001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('combmod')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute combmod
print("Running combmod...")
dm_cmd.combmod(
    proto_i='t_mod1',  # required
    inmods_i=['optional'],  # required
    modelout_o='t_combmod_out',  # required
    # tolernce_p=0.001,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("combmod execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_combmod_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")